# Structured Streaming — 04: Joins & Deduplication

Three join patterns: stream-static (most common), stream-stream (advanced), stream self-join.

- **Stream-static join**: stream is enriched with a static lookup table; static DF is broadcast.
- **Stream-stream join**: both sides are streams; requires watermarks on both sides.
- **Deduplication**: `dropDuplicates(["key_col"])` within a watermark window.

## Setup

In [ ]:
import tempfile, os, time, threading, csv, random, shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, expr, window,
    count, when, broadcast
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("SS-04-Joins")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

BASE = tempfile.mkdtemp().replace('\\', '/') + '/ss04'
os.makedirs(BASE, exist_ok=True)
print("BASE:", BASE)

## 1. Stream–Static Join

Most common pattern: enrich a live stream with a reference/lookup table.

- Static DF is loaded once at the start (can be re-read periodically via `foreachBatch` for slowly-changing dims).
- Broadcast the static side to avoid shuffle on the stream side.
- Any join type supported (inner, left, right, anti).

In [ ]:
# Static product lookup table
PRODUCT_DATA = [
    (1, "Laptop",    1200.00, "Electronics"),
    (2, "Mouse",       25.00, "Electronics"),
    (3, "Keyboard",    75.00, "Electronics"),
    (4, "Desk",       300.00, "Furniture"),
    (5, "Chair",      250.00, "Furniture"),
    (6, "Monitor",    400.00, "Electronics"),
    (7, "Headset",     90.00, "Electronics"),
    (8, "Webcam",      60.00, "Electronics"),
]
product_schema = StructType([
    StructField("product_id",   IntegerType(), False),
    StructField("product_name", StringType(),  False),
    StructField("unit_price",   DoubleType(),  False),
    StructField("category",     StringType(),  False),
])
products_df = spark.createDataFrame(PRODUCT_DATA, product_schema)
products_df.show()

# Streaming orders: rate source generates order_id (value), product_id derived from modulo
CKPT_SS_STATIC = BASE + '/ckpt_static_join'

orders_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 3)
    .load()
    .withColumn("order_id",   col("value"))
    .withColumn("product_id", ((col("value") % 8) + 1).cast("int"))
    .withColumn("quantity",   ((col("value") % 5) + 1).cast("int"))
    .withColumn("order_time", current_timestamp())
    .drop("value")
)

# Enrich: join stream with static lookup
enriched = (
    orders_stream
    .join(broadcast(products_df), on="product_id", how="inner")
    .withColumn("line_total", col("quantity") * col("unit_price"))
    .select("order_id", "order_time", "product_id", "product_name", "category", "quantity", "unit_price", "line_total")
)

q_static = (
    enriched
    .writeStream
    .format("memory")
    .queryName("enriched_orders")
    .outputMode("append")
    .option("checkpointLocation", CKPT_SS_STATIC)
    .start()
)

print("Stream-static join running. Waiting 15 seconds...")
time.sleep(15)
print("\nEnriched orders sample:")
spark.sql("SELECT * FROM enriched_orders ORDER BY order_id LIMIT 15").show(truncate=False)
print("Total enriched rows:", spark.sql("SELECT COUNT(*) FROM enriched_orders").first()[0])
q_static.stop()

## 2. Stream–Static Left Anti Join (Blacklist Filtering)

- **Left anti join**: keep stream rows that do NOT match the static set.
- Use case: filtering events against a blocklist, excluding cancelled orders, remove known bots.
- The static DF here is a "blocklist" of product_ids to suppress.

In [ ]:
CKPT_ANTI = BASE + '/ckpt_anti'

# Blocklist: suppress orders for products 2 (Mouse) and 4 (Desk)
blocklist = spark.createDataFrame([(2,), (4,)], ["product_id"])

filtered_stream = (
    orders_stream
    .join(blocklist, on="product_id", how="left_anti")
    .withColumn("order_time", current_timestamp())
    .select("order_id", "product_id", "order_time")
)

q_anti = (
    filtered_stream
    .writeStream
    .format("memory")
    .queryName("filtered_orders")
    .outputMode("append")
    .option("checkpointLocation", CKPT_ANTI)
    .start()
)

time.sleep(12)
print("Filtered orders (product_id 2 and 4 excluded):")
spark.sql("SELECT product_id, COUNT(*) as cnt FROM filtered_orders GROUP BY product_id ORDER BY product_id").show()
q_anti.stop()

## 3. Stream–Stream Join

- Both sides are live streams; Spark must buffer state from both until a match is found.
- **REQUIRES watermarks on BOTH streams** to bound state.
- Watermark constraint: the join condition must include an event-time range (e.g. within 30 seconds).
- Only inner join and left outer join are supported for stream-stream.
- Higher latency than stream-static because both sides must buffer.

In [ ]:
CKPT_SS_STREAM = BASE + '/ckpt_stream_stream'

# Stream A: order events
orders_a = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
    .withColumn("order_id",    col("value"))
    .withColumn("order_time",  current_timestamp())
    .withColumn("customer",    lit("cust_") )
    .drop("value")
    .withWatermark("order_time", "20 seconds")
)

# Stream B: payment confirmations (offset by small delay, same key space)
payments_b = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
    .withColumn("payment_order_id", col("value"))
    .withColumn("payment_time",     current_timestamp())
    .withColumn("status",           lit("CONFIRMED"))
    .drop("value")
    .withWatermark("payment_time", "20 seconds")
)

# Join on matching order_id, within a 30-second event-time window
joined_streams = (
    orders_a.alias("o")
    .join(
        payments_b.alias("p"),
        expr("""
            o.order_id = p.payment_order_id AND
            p.payment_time BETWEEN o.order_time AND o.order_time + interval 30 seconds
        """),
        how="inner"
    )
    .select(
        col("o.order_id"),
        col("o.order_time"),
        col("p.payment_time"),
        col("p.status")
    )
)

q_stream_stream = (
    joined_streams
    .writeStream
    .format("memory")
    .queryName("order_payments")
    .outputMode("append")
    .option("checkpointLocation", CKPT_SS_STREAM)
    .start()
)

print("Stream-stream join running. Waiting 30 seconds...")
time.sleep(30)
matched = spark.sql("SELECT COUNT(*) AS matched FROM order_payments").first()["matched"]
print(f"Matched order-payment pairs: {matched}")
spark.sql("SELECT * FROM order_payments ORDER BY order_id LIMIT 10").show(truncate=False)
q_stream_stream.stop()

## 4. dropDuplicates — Deduplication on a Stream

- Duplicate events are common in streaming (at-least-once delivery, retries, duplicated producers).
- `dropDuplicates(["event_id"])` keeps only the first occurrence of each key.
- **Without watermark**: state kept forever (safe for low-cardinality keys, unbounded memory otherwise).
- **With watermark**: state cleared after watermark passes — can miss late duplicates outside the window.
- `dropDuplicates` must appear **BEFORE** any aggregation.

In [ ]:
CKPT_DEDUP = BASE + '/ckpt_dedup'

# Simulate duplicates: every even value is sent twice (value and value-1 share same "event_id")
raw_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 4)
    .load()
    .withColumn("event_time", current_timestamp())
    .withColumn("event_id",   (col("value") / 2).cast("long"))   # duplicates: consecutive pairs share same id
    .withColumn("payload",    col("value"))
)

# Raw: count without dedup
q_raw = (
    raw_stream
    .writeStream
    .format("memory")
    .queryName("raw_events")
    .outputMode("append")
    .option("checkpointLocation", CKPT_DEDUP + '_raw')
    .start()
)

# Deduped: keep only first event_id within 30-second watermark window
deduped_stream = (
    raw_stream
    .withWatermark("event_time", "30 seconds")
    .dropDuplicates(["event_id"])
)

q_dedup = (
    deduped_stream
    .writeStream
    .format("memory")
    .queryName("deduped_events")
    .outputMode("append")
    .option("checkpointLocation", CKPT_DEDUP + '_dedup')
    .start()
)

print("Deduplication demo running. Waiting 20 seconds...")
time.sleep(20)

raw_cnt   = spark.sql("SELECT COUNT(*) AS n FROM raw_events").first()["n"]
dedup_cnt = spark.sql("SELECT COUNT(*) AS n FROM deduped_events").first()["n"]
dedup_ids = spark.sql("SELECT COUNT(DISTINCT event_id) AS n FROM raw_events").first()["n"]

print(f"\nRaw events : {raw_cnt}")
print(f"Unique IDs : {dedup_ids}")
print(f"After dedup: {dedup_cnt} (should be ~{dedup_ids})")
print(f"Duplicates removed: {raw_cnt - dedup_cnt}")

q_raw.stop()
q_dedup.stop()

## 5. foreachBatch — Refreshing the Static Side

- **Problem**: static DF is broadcast once; if the reference table changes, the stream doesn't pick up updates.
- **Solution**: use `foreachBatch` to re-read the lookup table every micro-batch.
- `foreachBatch(func)` gives you a plain batch DataFrame each trigger — write any batch logic.
- This is the escape hatch for anything the streaming API doesn't support natively.

In [ ]:
CKPT_FOREACH = BASE + '/ckpt_foreach'
OUTPUT_FOREACH = BASE + '/foreach_output'
os.makedirs(OUTPUT_FOREACH, exist_ok=True)

results_store = []   # accumulate results across batches for display

def enrich_and_count(batch_df, batch_id):
    if batch_df.count() == 0:
        return
    enriched_b = batch_df.join(broadcast(products_df), on="product_id", how="inner")
    by_cat = enriched_b.groupBy("category").count()
    rows = [(row["category"], row["count"], batch_id) for row in by_cat.collect()]
    results_store.extend(rows)
    print(f"  [Batch {batch_id}] categories: {[(r[0], r[1]) for r in rows]}")

q_foreach = (
    orders_stream
    .writeStream
    .foreachBatch(enrich_and_count)
    .option("checkpointLocation", CKPT_FOREACH)
    .start()
)

print("foreachBatch query running. Waiting 18 seconds...")
time.sleep(18)
q_foreach.stop()

print("\nSummary across all batches:")
summary = {}
for cat, cnt, bid in results_store:
    summary[cat] = summary.get(cat, 0) + cnt
for cat, total in sorted(summary.items()):
    print(f"  {cat}: {total}")

## Key Takeaways

- **Stream-static join**: cheapest pattern — broadcast the static side, works with any join type.
- **Stream-stream join**: both sides buffer state — always watermark both, use time-range conditions.
- **Left anti join** on a blocklist: elegantly filters known-bad keys from a live stream.
- **`dropDuplicates`** before aggregation removes duplicate events; pair with watermark to bound state.
- **`foreachBatch`** breaks out of the streaming API when you need full batch flexibility per trigger.

## Practice Exercises

1. Change the stream-static join to `left` (outer) — observe NULLs for unmatched product_ids.
2. Add a second product to the blocklist and verify it's excluded.
3. Increase the dedup watermark to 60 seconds — does the dedup count change?
4. In `foreachBatch`, write the enriched batch to Parquet instead of collecting to a list.

## Teardown

In [ ]:
shutil.rmtree(BASE, ignore_errors=True)
print("Temp directories cleaned up.")
spark.stop()
print("SparkSession stopped.")